In [2]:
# Standard library imports
# List files, simple math
import glob
import math
from collections import defaultdict, Counter


# Array and dataframe handling libraries
import numpy as np
import pandas as pd

# Adds loading/processing bar, to keep you informed
from tqdm.auto import tqdm

In [3]:
# =============================================================================
# GENOMAD SUMMARY PROCESSING
# =============================================================================
# Process GeNomad viral summary files and extract key metrics.
# Only contigs with at least 1 hallmark viral gene are considered.
# =============================================================================

import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

INPUT_PATTERN = 'data/genomad_filtered/*/*/final.contigs_virus_summary.tsv'

# -----------------------------------------------------------------------------
# Data containers
# -----------------------------------------------------------------------------

RES = {}                  # Sample-level summary metrics
contig_lengths = {}           # Length per contig (seq_name -> length)
contig_taxonomy = {}          # Taxonomy per contig (seq_name -> taxonomy)
genetic_code_counts = {}      # Genetic code distribution per sample

# -----------------------------------------------------------------------------
# Main processing loop
# -----------------------------------------------------------------------------

files = glob.glob(INPUT_PATTERN)

for file_path in tqdm(files, desc='Processing GeNomad summaries'):
    # Extract sample ID from file path (e.g., "SRR123" from ".../SRR123_xxx/...")
    sample_id = file_path.split('/')[2].split('_')[0]
    
    # Read the summary file
    df = pd.read_table(file_path)
    
    # Filter: keep only contigs with at least 1 hallmark viral gene
    df_filtered = df[df['n_hallmarks'] > 0].copy()
    
    # Store contig-level taxonomy
    contig_taxonomy.update(
        df_filtered.set_index('seq_name')['taxonomy'].to_dict()
    )
    
    # Store contig-level lengths
    contig_lengths.update(
        df_filtered.set_index('seq_name')['length'].to_dict()
    )
    
    # Compute sample-level summary statistics
    RES[sample_id] = {
        'Num. Unique Viral Genomes/Contigs': df_filtered.shape[0],
        'Genes/Contig': df_filtered['n_genes'].median(),
        'Length/Contig': df_filtered['length'].median(),
        'Max Genes Length/Contig': df_filtered['n_genes'].max(),
        'Max Length/Contig': df_filtered['length'].max(),
        'Unq genetic_code': df_filtered['genetic_code'].nunique(),
        'n_hallmarks': df_filtered['n_hallmarks'].sum(),
    }
    
    # Store genetic code distribution (as percentage)
    genetic_code_counts[sample_id] = (
        df_filtered['genetic_code'].value_counts(normalize=True) * 100
    )

Processing GeNomad summaries: 100%|████████████████████████████████████████████| 4/4 [00:00<00:00, 159.50it/s]


In [4]:
NUM = pd.DataFrame(RES).T
NUM.head(2)

,Num. Unique Viral Genomes/Contigs,Genes/Contig,Length/Contig,Max Genes Length/Contig,Max Length/Contig,Unq genetic_code,n_hallmarks
HPV401,102.0,2.0,949.0,137.0,96807.0,2.0,159.0
HPV402,5.0,3.0,1406.0,24.0,18088.0,1.0,12.0


In [5]:
# =============================================================================
# PHAROKKA ANNOTATION PROCESSING
# =============================================================================
# Process Pharokka gene annotations, keeping only contigs that have 
# GeNomad hallmark viral genes.
# =============================================================================

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

INPUT_PATTERN = 'data/pharokka_output/*/pharokka_cds_final_merged_output.tsv'
USECOLS = ['contig', 'annot', 'category']

# -----------------------------------------------------------------------------
# Data containers
# -----------------------------------------------------------------------------

annotation_categories = {}                # annot -> category mapping
results_by_sample = []                    # Sample-level aggregated results
# -----------------------------------------------------------------------------
# Main processing loop
# -----------------------------------------------------------------------------

files = glob.glob(INPUT_PATTERN)

for file_path in tqdm(files, desc='Processing Pharokka annotations'):
    # Extract sample ID from file path
    sample_id = file_path.split('/')[2].split('_')[0]
    
    # Read annotation file
    df = pd.read_table(file_path)
    
    # Filter out unknown function annotations
    df_filtered = df[df['category'] != 'unknown function'].copy()
    
    # Keep only relevant columns
    df_filtered = df_filtered[USECOLS]
    
    # Update annotation-to-category mapping
    annotation_categories.update(
        df_filtered.set_index('annot')['category'].to_dict()
    )
    
    # Add contig-level information from GeNomad ---
    
    # Add taxonomy for each contig (from previously processed GeNomad data)
    df_filtered['taxonomy'] = df_filtered['contig'].map(contig_taxonomy)
    # Add contig length (from previously processed GeNomad data)
    df_filtered['contig_len'] = df_filtered['contig'].map(contig_lengths)
    # Passed GeNomad filtration?
    df_filtered = df_filtered[df_filtered.taxonomy.notna()]
    
    
    # --- Sample-level aggregation: taxonomy x annotation counts ---
    
    # Group by taxonomy and count annotations
    sample_aggregated = (
        df_filtered.groupby('taxonomy')['annot']
        .value_counts()
        .reset_index()
    )
    sample_aggregated['ID'] = sample_id
    
    results_by_sample.append(sample_aggregated)

# -----------------------------------------------------------------------------
# Combine results across all samples
# -----------------------------------------------------------------------------

# Concatenate all sample-level results
results_all_samples = pd.concat(results_by_sample, ignore_index=True)
results_all_samples = results_all_samples.rename(columns={'annot': 'category'})

Processing Pharokka annotations: 100%|██████████████████████████████████████████| 4/4 [00:00<00:00, 74.80it/s]


In [6]:
results_all_samples.head(2)

,taxonomy,category,count,ID
0,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,integrase,30,HPV401
1,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,terminase large subunit,15,HPV401


In [7]:
# Viral gene count table
CAT_PIVOT = pd.pivot_table(results_all_samples,
                           index='ID',
                           columns='category',
                           values='count',
                           aggfunc=np.sum).fillna(0)

CAT_PIVOT

category,ATPase,DNA binding protein,DNA helicase,DNA invertase,DNA methyltransferase,DNA polymerase,DNA primase,DNA transposition protein,DarB-like antirestriction,DnaB-like replicative helicase,...,terminase large subunit,terminase small subunit,toxin,toxin-antitoxin system HicB-like,transcriptional activator,transcriptional regulator,transcriptional repressor,transposase,upper collar connector,virion structural protein
ID,,,,,,,,,,,,,,,,,,,,,
HPV401,1.0,1.0,1.0,1.0,10.0,3.0,3.0,1.0,3.0,3.0,...,15.0,8.0,0.0,0.0,0.0,4.0,3.0,2.0,1.0,4.0
HPV402,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0
HPV403,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HPV404,0.0,0.0,3.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,2.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [8]:
# Viral gene annotation to category
# annotation_categories